In [5]:
import pandas as pd
import numpy as np
import sys, json, joblib, warnings

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore', category=UserWarning)

from config import get_var
token = get_var()

# Drive Config

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# HuggingFace Config

### 1-login

In [ ]:
from huggingface_hub import notebook_login
from huggingface_hub import hf_hub_download
from huggingface_hub import HfApi, create_repo

notebook_login()

In [ ]:
repo_id = "Meliodas-10/portnet-model"
create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

### 2-import best_params and model

In [ ]:
params_file = hf_hub_download(repo_id=repo_id, filename="best_params.json")
with open(params_file, "r") as f:
    best_params = json.load(f)
    
print("paramètres rechargés depuis Hugging Face !")

In [ ]:
model_file = hf_hub_download(repo_id=repo_id, filename="modele_portnet.pkl")
modele_final = joblib.load(model_file)

print("Modèle rechargé depuis Hugging Face !")

# Dataset Config

In [8]:
root = '/content/drive/MyDrive/Portnet Imputation Prediction'
pipeline = root + "/src"

raw_data = pd.read_csv(f'{root}/data.dsv', sep=';')
if pipeline not in sys.path: sys.path.append(pipeline)

from data_pipeline import portnet_pipeline

df = raw_data[raw_data['QTE_IMPUTE'] > 0].copy()
df = df[df['DEVISE'].isin(['EUR', 'USD'])]

X = df.drop(columns=['QTE_IMPUTE'])
y = df['QTE_IMPUTE']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


X_train_sample = X_train.sample(n=200000, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

# 1- Algorithm Selection and Training

## 1.1- Dummy Model

In [10]:
from sklearn.dummy import DummyRegressor

model = TransformedTargetRegressor(
    regressor=DummyRegressor(strategy='mean'),
    func=np.log1p,
    inverse_func=np.expm1
)

model.fit(X_train_sample, y_train_sample)

y_pred_baseline = model.predict(X_test)

mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_base = r2_score(y_test, y_pred_baseline)

print("--- BASELINE Score ---")
print(f"MAE  : {mae_base:.2f}")
print(f"RMSE : {rmse_base:.2f}")
print(f"R²   : {r2_base:.4f}")

--- BASELINE Score ---
MAE  : 113029.09
RMSE : 2484896.59
R²   : -0.0020


## 1.2- Models CV Loop

In [11]:
algorithmes = {
    "Random Forest (Bagging)": RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    "HistGradientBoosting (Boosting)": HistGradientBoostingRegressor(random_state=42)
}

In [12]:
test_slice = X_train.head(5)
target_slice = y_train.head(5)

print("Testing pipeline on a small slice...")
modele_complet = TransformedTargetRegressor(
    regressor=portnet_pipeline.set_params(algorithme=algorithmes["Random Forest (Bagging)"]),
    func=np.log1p,
    inverse_func=np.expm1
)
modele_complet.fit(test_slice, target_slice)
print("\n----- Sanity check passed successfully! The pipeline fits without errors -----")

Testing pipeline on a small slice...
[_drop_unnecessary_columns] exécuté en 0.001s | Lignes restantes: 5
[_clean_quantite_domicile] exécuté en 0.001s | Lignes restantes: 5
[_set_datatypes] exécuté en 0.008s | Lignes restantes: 5
[_optimize_memory] exécuté en 0.003s | Lignes restantes: 5
[_delai_extracting] exécuté en 0.002s | Lignes restantes: 5
[_strategic_grouping] exécuté en 0.001s | Lignes restantes: 5
[_temporal_engineering] exécuté en 0.002s | Lignes restantes: 5
[_flags_creation] exécuté en 0.001s | Lignes restantes: 5
[_unify_currency_to_eur] exécuté en 0.005s | Lignes restantes: 5
[_logarithmic_transform] exécuté en 0.001s | Lignes restantes: 5
[_impute_and_scale] exécuté en 0.002s | Lignes restantes: 5
[_drop_useless_text] exécuté en 0.000s | Lignes restantes: 5
[_target_encode] exécuté en 0.001s | Lignes restantes: 5

----- Sanity check passed successfully! The pipeline fits without errors -----


In [ ]:
X_train_sample = X_train.sample(n=200000, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

for nom, algo in algorithmes.items():
    print(f"\n--- Entraînement de {nom} en cours (CV = 5) ---")
    
    # Injection de l'algorithme dans le pipeline
    portnet_pipeline.set_params(algorithme=algo)
    
    # Bouclier logarithmique sur la cible y
    modele_complet = TransformedTargetRegressor(
        regressor=portnet_pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )
    
    # Validation croisée
    scores = cross_validate(
        modele_complet, 
        X_train_sample, 
        y_train_sample, 
        cv=5, 
        scoring=('neg_mean_absolute_error', 'r2'),
        n_jobs=1
    )
    
    # Extraction et affichage
    mae_moyen = -scores['test_neg_mean_absolute_error'].mean()
    r2_moyen = scores['test_r2'].mean()
    
    print(f"[{nom}] R² Moyen  : {r2_moyen:.4f}")
    print(f"[{nom}] MAE Moyen : {mae_moyen:.2f} unités")

## Results:
Random Forest (Bagging):
* R² Moyen  : 0.6608
* MAE Moyen : 35725.30 unités


HistGradientBoosting (Boosting):
* R² Moyen  : 0.5146
* MAE Moyen : 53058.07 unités

### --> Decision : Random Forest

# 2- Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

portnet_pipeline.set_params(algorithme=algorithmes["Random Forest (Bagging)"])

model = TransformedTargetRegressor(
    regressor=portnet_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param_distributions = {
    'regressor__algorithme__n_estimators': [100, 200, 300],
    'regressor__algorithme__max_depth': [15, 25, None],
    'regressor__algorithme__min_samples_split': [2, 5, 10],
    'regressor__algorithme__min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=5,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=1
)

print("--- Début de l'optimisation des hyperparamètres ---")
random_search.fit(X_train_sample, y_train_sample)

print(f"\nMeilleurs paramètres trouvés : {random_search.best_params_}")
print(f"Meilleur score R² (CV) : {random_search.best_score_:.4f}")

best_params = {k.replace('regressor__', ''): v for k, v in random_search.best_params_.items()}


with open("best_params.json", "w") as f:
    json.dump(best_params, f)

api = HfApi()
api.upload_file(
    path_or_fileobj="best_params.json",
    path_in_repo="best_params.json",
    repo_id=repo_id,
    repo_type="model"
)

print("Paramètres sauvegardés avec succès sur Hugging Face !")

# 3- Model Execution

In [ ]:
portnet_pipeline.set_params(**best_params)

full_model = TransformedTargetRegressor(
    regressor=portnet_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print("\n--- Entraînement du modèle final sur l'intégralité du dataset ---")
full_model.fit(X_train, y_train)

In [ ]:
y_pred = full_model.predict(X_test)

mae_base = mean_absolute_error(y_test, y_pred)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred))
r2_base = r2_score(y_test, y_pred)

print("--- Final Score ---")
print(f"MAE  : {mae_base:.2f}")
print(f"RMSE : {rmse_base:.2f}")
print(f"R²   : {r2_base:.4f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = full_model.predict(X_test)
final_r2 = r2_score(y_test, y_pred)
final_mae = mean_absolute_error(y_test, y_pred)

print(f"\n[Évaluation Finale sur le Test Set]")
print(f"R² Final : {final_r2:.4f}")
print(f"MAE Final : {final_mae:.2f} unités")

In [ ]:
joblib.dump(modele_final, "modele_portnet_final.pkl")
api = HfApi()

api.upload_file(
    path_or_fileobj="modele_portnet_final.pkl",
    path_in_repo="modele_portnet_final.pkl",
    repo_id=repo_id,
    repo_type="model"
)
print("modèle sauvegardés avec succès sur Hugging Face !")